<a href="https://colab.research.google.com/github/idarapatrick/Dual-Encoder-Model-for-Sahel-Dust-Forecasting/blob/main/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SahelDust Preprocessing Pipeline

This notebook extracts 72-hour ERA5 hourly sequences, same-day SMAP soil moisture, and MODIS AOD labels from Google Earth Engine, then normalizes and saves the aligned dataset.

## Data sources
- ERA5 hourly (ECMWF/ERA5/HOURLY): u10, v10, t2m, surface pressure
- SMAP soil moisture AM (NASA/SMAP/SPL3SMP_E/005 and 006)
- MODIS AOD (MODIS/061/MCD19A2_GRANULES): Optical_Depth_055
- AERONET proxy: MODIS AOD at 5 ground station locations

## Temporal window
- T-60 to T+12 hours relative to event day midnight
- SMAP uses same-day AM retrieval (06:00 local, before emission window)

## Temporal split
- Training: 2015-2022 (batches 001-086)
- Validation: 2023 (batches 087-097)
- Test: 2024 full unsubsampled (saheldust_test_full folder)

In [ ]:
!earthengine authenticate

# SETUP

In [ ]:
!pip install earthengine-api -q

import ee
import numpy as np
import os
import json
import time
import glob
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.preprocessing import StandardScaler
import joblib

ee.Initialize(project='sahel-dust-forecasting')

from google.colab import drive
drive.mount('/content/drive')

print("Ready")

# CONSTANTS

In [ ]:
SAHEL = ee.Geometry.Rectangle([-18, 10, 25, 25])

TARGET_H, TARGET_W = 68, 172
N_PIXELS = TARGET_H * TARGET_W

GRID = {
    'dimensions': {'width': TARGET_W, 'height': TARGET_H},
    'affineTransform': {
        'scaleX': 0.25, 'shearX': 0, 'translateX': -18.125,
        'shearY': 0, 'scaleY': -0.25, 'translateY': 25.125
    },
    'crsCode': 'EPSG:4326'
}

lons_1d = np.linspace(-18, 25, TARGET_W)
lats_1d = np.linspace(25, 10, TARGET_H)
LON_GRID, LAT_GRID = np.meshgrid(lons_1d, lats_1d)

BASE_DIR = '/content/drive/MyDrive/saheldust_data'
OUT_DIR = f'{BASE_DIR}/saheldust_processed_enriched'
os.makedirs(OUT_DIR, exist_ok=True)
CKPT_FILE = f'{OUT_DIR}/checkpoint.json'

NON_EVENT_RATE = 0.11
N_WORKERS = 4
SAVE_EVERY_N_DAYS = 30

ERA5_BANDS = [
    'u_component_of_wind_10m',
    'v_component_of_wind_10m',
    'temperature_2m',
    'surface_pressure',
    'boundary_layer_height',
    'total_precipitation',
    'dewpoint_temperature_2m'
]
N_ATM_VARS = len(ERA5_BANDS)
N_SURF_VARS = 7

print(f"Grid: {TARGET_H} x {TARGET_W} = {N_PIXELS:,} pixels")
print(f"ERA5 variables ({N_ATM_VARS}): {', '.join(ERA5_BANDS)}")
print(f"Surface features ({N_SURF_VARS}): soil_moisture, vegetation_water_content, prev_day_aod, lat, lon, month_sin, month_cos")
print(f"Non-event retention: {NON_EVENT_RATE*100:.0f}%")
print(f"Output: {OUT_DIR}")

# GEE Extraction Functions

In [ ]:
def fetch_pixels(ee_image):
    try:
        return ee.data.computePixels({
            'expression': ee_image,
            'fileFormat': 'NUMPY_NDARRAY',
            'grid': GRID
        })
    except Exception as e:
        return None


def get_era5_window(day):
    """
    Fetch 72-hour ERA5 window with 7 atmospheric variables.
    Window: T-60 to T+12 relative to event day midnight.
    Returns shape (H, W, 72, 7) or None.
    """
    t0 = ee.Date(day.strftime('%Y-%m-%dT00:00:00'))
    t_start = t0.advance(-60, 'hour')
    t_end = t0.advance(12, 'hour')

    era5 = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
              .filterDate(t_start, t_end)
              .select(ERA5_BANDS))

    n = era5.size().getInfo()
    if n < 72:
        return None

    era5 = ee.ImageCollection(era5.sort('system:time_start').toList(72))

    prefixes = ['u', 'v', 't', 's', 'b', 'p', 'd']
    band_images = []
    for band_name, prefix in zip(ERA5_BANDS, prefixes):
        bands = era5.select(band_name).toBands()
        bands = bands.rename([f'{prefix}_{i}' for i in range(72)])
        band_images.append(bands)

    combined = band_images[0]
    for bi in band_images[1:]:
        combined = combined.addBands(bi)

    pixels = fetch_pixels(combined)
    if pixels is None:
        return None

    arrays = []
    for prefix in prefixes:
        arr = np.stack([pixels[f'{prefix}_{i}'] for i in range(72)], axis=0)
        arrays.append(arr)

    era5_array = np.stack(arrays, axis=-1).astype(np.float32)
    return era5_array.transpose(1, 2, 0, 3)


def get_smap_day(day):
    """
    Fetch same-day SMAP AM soil moisture and vegetation water content.
    Returns tuple of (soil_moisture, veg_water_content) arrays, each shape (H, W).
    Returns (None, None) if unavailable.
    """
    coll_id = ('NASA/SMAP/SPL3SMP_E/005'
               if day < datetime(2023, 12, 4)
               else 'NASA/SMAP/SPL3SMP_E/006')

    day_start = ee.Date(day.strftime('%Y-%m-%d'))
    day_end = day_start.advance(1, 'day')

    smap = (ee.ImageCollection(coll_id)
              .filterDate(day_start, day_end)
              .select(['soil_moisture_am', 'vegetation_water_content_am']))

    if smap.size().getInfo() == 0:
        return None, None

    img = smap.first()

    sm_pixels = fetch_pixels(img.select('soil_moisture_am'))
    if sm_pixels is None:
        return None, None
    sm = sm_pixels['soil_moisture_am'].astype(np.float32)
    sm[sm < 0] = np.nan

    vwc_pixels = fetch_pixels(img.select('vegetation_water_content_am'))
    if vwc_pixels is None:
        return None, None
    vwc = vwc_pixels['vegetation_water_content_am'].astype(np.float32)
    vwc[vwc < 0] = np.nan

    return sm, vwc


def get_modis_aod(day):
    """
    Fetch MODIS AOD for a single day.
    Returns shape (H, W) with actual AOD values (scaled), or None.
    """
    day_start = ee.Date(day.strftime('%Y-%m-%d'))
    day_end = day_start.advance(1, 'day')

    modis = (ee.ImageCollection('MODIS/061/MCD19A2_GRANULES')
               .filterDate(day_start, day_end)
               .select('Optical_Depth_055'))

    if modis.size().getInfo() == 0:
        return None

    pixels = fetch_pixels(modis.max())
    if pixels is None:
        return None

    raw = pixels['Optical_Depth_055'].astype(np.float32)
    aod = raw * 0.001
    aod[raw <= 0] = np.nan
    return aod

# Per-day Processing

In [ ]:
def process_day(day):
    """
    Process one day. Returns list of (atm, surf, label, year) tuples.

    Atmospheric input: (72, 7) - u10, v10, t2m, sp, blh, tp, d2m
    Surface input: (7,) - soil_moisture, veg_water_content, prev_day_aod,
                          lat, lon, month_sin, month_cos
    Label: binary from same-day MODIS AOD > 0.5
    """
    try:
        today_aod = get_modis_aod(day)
        if today_aod is None:
            return day, []

        prev_day = day - timedelta(days=1)
        prev_aod = get_modis_aod(prev_day)
        if prev_aod is None:
            prev_aod = np.zeros((TARGET_H, TARGET_W), dtype=np.float32)

        smap_sm, smap_vwc = get_smap_day(day)
        if smap_sm is None:
            return day, []

        era5_grid = get_era5_window(day)
        if era5_grid is None:
            return day, []

        label_grid = (today_aod > 0.5).astype(np.float32)
        label_grid[np.isnan(today_aod)] = np.nan

        month = day.month
        month_sin = np.float32(np.sin(2 * np.pi * month / 12))
        month_cos = np.float32(np.cos(2 * np.pi * month / 12))

        samples = []
        rng = np.random.default_rng(seed=int(day.strftime('%Y%m%d')))

        for row in range(TARGET_H):
            for col in range(TARGET_W):
                atm = era5_grid[row, col]
                sm = smap_sm[row, col]
                vwc = smap_vwc[row, col]
                lbl = label_grid[row, col]
                paod = prev_aod[row, col]

                if np.isnan(sm) or np.isnan(lbl) or np.any(np.isnan(atm)):
                    continue
                if np.isnan(paod):
                    paod = 0.0
                if np.isnan(vwc):
                    vwc = 0.0

                if lbl == 0.0 and rng.random() > NON_EVENT_RATE:
                    continue

                lat = np.float32(LAT_GRID[row, col])
                lon = np.float32(LON_GRID[row, col])

                surf = np.array(
                    [sm, vwc, paod, lat, lon, month_sin, month_cos],
                    dtype=np.float32
                )
                samples.append((atm.copy(), surf, np.float32(lbl), np.int16(day.year)))

        return day, samples

    except Exception as e:
        return day, []

# Checkpoint and Batch Saving

In [ ]:
def load_checkpoint():
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE, 'r') as f:
            return json.load(f)
    return {'last_date': None, 'total_samples': 0, 'next_batch': 1}


def save_checkpoint(last_date, total_samples, next_batch):
    with open(CKPT_FILE, 'w') as f:
        json.dump({
            'last_date': last_date.strftime('%Y-%m-%d'),
            'total_samples': total_samples,
            'next_batch': next_batch
        }, f)
    print(f"  Checkpoint: {last_date.strftime('%Y-%m-%d')} | samples={total_samples:,} | batch={next_batch}")


def flush_batch(atm_buf, surf_buf, label_buf, year_buf, batch_idx):
    if not atm_buf:
        return [], [], [], []
    path = f'{OUT_DIR}/batch_{batch_idx:04d}.npz'
    np.savez_compressed(
        path,
        X_atm=np.array(atm_buf, dtype=np.float32),
        X_surf=np.array(surf_buf, dtype=np.float32),
        y=np.array(label_buf, dtype=np.float32),
        years=np.array(year_buf, dtype=np.int16)
    )
    size_mb = os.path.getsize(path) / 1e6
    print(f"  Batch {batch_idx:04d}: {len(atm_buf):,} samples ({size_mb:.1f} MB)")
    return [], [], [], []

# Main Processing Loop (2015-2023)

In [ ]:
np.random.seed(42)

all_days = []
d = datetime(2015, 4, 1)
while d <= datetime(2023, 12, 31):
    all_days.append(d)
    d += timedelta(days=1)

print(f"Total days to process: {len(all_days):,}")

ckpt = load_checkpoint()
if ckpt['last_date']:
    resume_from = datetime.strptime(ckpt['last_date'], '%Y-%m-%d')
    all_days = [d for d in all_days if d > resume_from]
    print(f"Resuming from: {ckpt['last_date']}")
    print(f"Remaining days: {len(all_days):,}")

total_samples = ckpt['total_samples']
next_batch = ckpt['next_batch']

atm_buf, surf_buf, label_buf, year_buf = [], [], [], []
days_since_flush = 0
n_skipped = 0
t_start_wall = time.time()

day_batches = [all_days[i:i+N_WORKERS] for i in range(0, len(all_days), N_WORKERS)]

for batch_idx_loop, day_batch in enumerate(day_batches):
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(process_day, day): day for day in day_batch}
        for future in as_completed(futures):
            day, samples = future.result()
            if not samples:
                n_skipped += 1
                continue
            for atm, surf, lbl, yr in samples:
                atm_buf.append(atm)
                surf_buf.append(surf)
                label_buf.append(lbl)
                year_buf.append(yr)
            total_samples += len(samples)
            days_since_flush += 1

    elapsed = (time.time() - t_start_wall) / 3600
    days_done = (batch_idx_loop + 1) * N_WORKERS

    print(f"[{day_batch[-1].strftime('%Y-%m-%d')}] "
          f"days={days_done:,} | samples={total_samples:,} | "
          f"buf={len(atm_buf):,} | elapsed={elapsed:.1f}h")

    if days_since_flush >= SAVE_EVERY_N_DAYS:
        atm_buf, surf_buf, label_buf, year_buf = flush_batch(
            atm_buf, surf_buf, label_buf, year_buf, next_batch
        )
        save_checkpoint(day_batch[-1], total_samples, next_batch + 1)
        next_batch += 1
        days_since_flush = 0

if atm_buf:
    flush_batch(atm_buf, surf_buf, label_buf, year_buf, next_batch)
    save_checkpoint(all_days[-1], total_samples, next_batch + 1)

print(f"\nDone. Total samples: {total_samples:,} | Skipped: {n_skipped:,}")

# Process 2024 Test Set (No Subsampling)

In [ ]:
print("PROCESSING 2024 TEST SET (NO SUBSAMPLING)")

TEST_DIR = f'{BASE_DIR}/saheldust_test_enriched'
os.makedirs(TEST_DIR, exist_ok=True)

def process_day_full(day):
    """Same as process_day but keeps ALL valid pixels."""
    try:
        today_aod = get_modis_aod(day)
        if today_aod is None:
            return day, []

        prev_day = day - timedelta(days=1)
        prev_aod = get_modis_aod(prev_day)
        if prev_aod is None:
            prev_aod = np.zeros((TARGET_H, TARGET_W), dtype=np.float32)

        smap_sm, smap_vwc = get_smap_day(day)
        if smap_sm is None:
            return day, []

        era5_grid = get_era5_window(day)
        if era5_grid is None:
            return day, []

        label_grid = (today_aod > 0.5).astype(np.float32)
        label_grid[np.isnan(today_aod)] = np.nan

        month = day.month
        month_sin = np.float32(np.sin(2 * np.pi * month / 12))
        month_cos = np.float32(np.cos(2 * np.pi * month / 12))

        samples = []
        for row in range(TARGET_H):
            for col in range(TARGET_W):
                atm = era5_grid[row, col]
                sm = smap_sm[row, col]
                vwc = smap_vwc[row, col]
                lbl = label_grid[row, col]
                paod = prev_aod[row, col]

                if np.isnan(sm) or np.isnan(lbl) or np.any(np.isnan(atm)):
                    continue
                if np.isnan(paod):
                    paod = 0.0
                if np.isnan(vwc):
                    vwc = 0.0

                lat = np.float32(LAT_GRID[row, col])
                lon = np.float32(LON_GRID[row, col])
                surf = np.array([sm, vwc, paod, lat, lon, month_sin, month_cos], dtype=np.float32)
                samples.append((atm.copy(), surf, np.float32(lbl), np.int16(day.year)))

        return day, samples
    except Exception as e:
        return day, []

test_days = []
d = datetime(2024, 1, 1)
while d <= datetime(2024, 12, 31):
    test_days.append(d)
    d += timedelta(days=1)

print(f"2024 test days: {len(test_days)}")

test_atm_buf, test_surf_buf, test_y_buf = [], [], []
n_test = 0
batch_test = 1
t_start = time.time()

day_batches_test = [test_days[i:i+N_WORKERS] for i in range(0, len(test_days), N_WORKERS)]

for bi, day_batch in enumerate(day_batches_test):
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(process_day_full, day): day for day in day_batch}
        for future in as_completed(futures):
            day, samples = future.result()
            if not samples:
                continue
            for atm, surf, lbl, yr in samples:
                test_atm_buf.append(atm)
                test_surf_buf.append(surf)
                test_y_buf.append(lbl)
            n_test += len(samples)

    elapsed = (time.time() - t_start) / 3600
    print(f"[{day_batch[-1].strftime('%Y-%m-%d')}] samples={n_test:,} | elapsed={elapsed:.2f}h")

    if (bi + 1) % (30 // N_WORKERS) == 0 and test_atm_buf:
        path = f'{TEST_DIR}/test_batch_{batch_test:03d}.npz'
        np.savez_compressed(path,
            X_atm=np.array(test_atm_buf, dtype=np.float32),
            X_surf=np.array(test_surf_buf, dtype=np.float32),
            y=np.array(test_y_buf, dtype=np.float32))
        print(f"  Saved {path}")
        test_atm_buf, test_surf_buf, test_y_buf = [], [], []
        batch_test += 1

if test_atm_buf:
    path = f'{TEST_DIR}/test_batch_{batch_test:03d}.npz'
    np.savez_compressed(path,
        X_atm=np.array(test_atm_buf, dtype=np.float32),
        X_surf=np.array(test_surf_buf, dtype=np.float32),
        y=np.array(test_y_buf, dtype=np.float32))
    print(f"Saved {path}")

print(f"\n2024 test set complete: {n_test:,} samples")

# Merge Train/Validation Batches

In [ ]:
print("Loading training and validation batches...")
batch_files = sorted(glob.glob(f'{OUT_DIR}/batch_*.npz'))
print(f"Found {len(batch_files)} batch files")

train_atm, train_surf, train_y = [], [], []
val_atm, val_surf, val_y = [], [], []

for bf in batch_files:
    data = np.load(bf)
    X_a = data['X_atm'].astype(np.float32)
    X_s = data['X_surf'].astype(np.float32)
    y_b = data['y'].astype(np.float32)
    yrs = data['years']
    data.close()

    train_mask = yrs <= 2022
    val_mask = yrs == 2023

    if train_mask.sum() > 0:
        train_atm.append(X_a[train_mask])
        train_surf.append(X_s[train_mask])
        train_y.append(y_b[train_mask])
    if val_mask.sum() > 0:
        val_atm.append(X_a[val_mask])
        val_surf.append(X_s[val_mask])
        val_y.append(y_b[val_mask])

    print(f"  {os.path.basename(bf)}: train={train_mask.sum():,} val={val_mask.sum():,}")

X_atm_tr = np.concatenate(train_atm); del train_atm
X_surf_tr = np.concatenate(train_surf); del train_surf
y_tr = np.concatenate(train_y); del train_y

X_atm_val = np.concatenate(val_atm); del val_atm
X_surf_val = np.concatenate(val_surf); del val_surf
y_val = np.concatenate(val_y); del val_y

print(f"\nTrain: {len(y_tr):,} samples | dust rate: {y_tr.mean()*100:.1f}%")
print(f"Val: {len(y_val):,} samples | dust rate: {y_val.mean()*100:.1f}%")
print(f"Atm shape: {X_atm_tr.shape}")
print(f"Surf shape: {X_surf_tr.shape}")

# Load Test Set

In [ ]:
TEST_DIR = f'{BASE_DIR}/saheldust_test_enriched'
print(f"TEST_DIR is set to: {TEST_DIR}")

In [ ]:
test_files = sorted(glob.glob(f'{TEST_DIR}/test_batch_*.npz'))
test_atm, test_surf, test_y = [], [], []

for tf in test_files:
    data = np.load(tf)
    test_atm.append(data['X_atm'].astype(np.float32))
    test_surf.append(data['X_surf'].astype(np.float32))
    test_y.append(data['y'].astype(np.float32))
    data.close()

X_atm_te = np.concatenate(test_atm); del test_atm
X_surf_te = np.concatenate(test_surf); del test_surf
y_te = np.concatenate(test_y); del test_y

print(f"Test: {len(y_te):,} samples | dust rate: {y_te.mean()*100:.1f}%")
print(f"Atm shape: {X_atm_te.shape}")
print(f"Surf shape: {X_surf_te.shape}")

# Normalize and Save

In [ ]:
# malloc trim + surface scaling

import ctypes
import gc
from sklearn.preprocessing import StandardScaler

ctypes.CDLL("libc.so.6").malloc_trim(0)

surf_scaler = StandardScaler()

X_surf_tr_n = surf_scaler.fit_transform(X_surf_tr).astype(np.float32)
del X_surf_tr
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
np.save(f'{OUT_DIR}/X_surf_train.npy', X_surf_tr_n)
del X_surf_tr_n
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)

X_surf_val_n = surf_scaler.transform(X_surf_val).astype(np.float32)
del X_surf_val
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
np.save(f'{OUT_DIR}/X_surf_val.npy', X_surf_val_n)
del X_surf_val_n
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)

X_surf_te_n = surf_scaler.transform(X_surf_te).astype(np.float32)
del X_surf_te
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
np.save(f'{OUT_DIR}/X_surf_test.npy', X_surf_te_n)
del X_surf_te_n
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)

joblib.dump(surf_scaler, f'{OUT_DIR}/surf_scaler.pkl')
print("Surface scaling done.")

In [ ]:
import ctypes
import gc
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

ctypes.CDLL("libc.so.6").malloc_trim(0)

atm_scaler = StandardScaler()
N_tr = len(y_tr)
CHUNK = 2000 # rows per chunk, lower this to 2000 if it still crashes

# ── Fit the scaler using partial_fit over chunks (no full array in memory) ────
n_chunks = int(np.ceil(N_tr / CHUNK))
for i in range(n_chunks):
    chunk = X_atm_tr[i*CHUNK:(i+1)*CHUNK].reshape(-1, N_ATM_VARS)
    atm_scaler.partial_fit(chunk)
    del chunk
    gc.collect()
print("partial_fit done.")

# ── Transform and save chunk by chunk directly to disk ────────────────────────
fp = np.lib.format.open_memmap(
    f'{OUT_DIR}/X_atm_train.npy',
    mode='w+',
    dtype=np.float32,
    shape=(N_tr, 72, N_ATM_VARS)
)

for i in range(n_chunks):
    chunk = X_atm_tr[i*CHUNK:(i+1)*CHUNK]
    n_rows = chunk.shape[0]
    fp[i*CHUNK:i*CHUNK+n_rows] = atm_scaler.transform(
        chunk.reshape(-1, N_ATM_VARS)
    ).reshape(n_rows, 72, N_ATM_VARS).astype(np.float32)
    del chunk
    gc.collect()

del fp
del X_atm_tr
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)

joblib.dump(atm_scaler, f'{OUT_DIR}/atm_scaler_partial.pkl')
print("Atm train done.")

In [ ]:
# atm val + test

import ctypes
import gc
import joblib

ctypes.CDLL("libc.so.6").malloc_trim(0)

atm_scaler = joblib.load(f'{OUT_DIR}/atm_scaler_partial.pkl')
N_val = len(y_val)
N_te  = len(y_te)

if not X_atm_val.flags['C_CONTIGUOUS']:
    X_atm_val = np.ascontiguousarray(X_atm_val)

X_atm_val_n = atm_scaler.transform(
    X_atm_val.reshape(-1, N_ATM_VARS)
).reshape(N_val, 72, N_ATM_VARS).astype(np.float32)
del X_atm_val
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
np.save(f'{OUT_DIR}/X_atm_val.npy', X_atm_val_n)
del X_atm_val_n
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
print("Atm val done.")

if not X_atm_te.flags['C_CONTIGUOUS']:
    X_atm_te = np.ascontiguousarray(X_atm_te)

X_atm_te_n = atm_scaler.transform(
    X_atm_te.reshape(-1, N_ATM_VARS)
).reshape(N_te, 72, N_ATM_VARS).astype(np.float32)
del X_atm_te
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
np.save(f'{OUT_DIR}/X_atm_test.npy', X_atm_te_n)
del X_atm_te_n
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
print("Atm test done.")

joblib.dump(atm_scaler, f'{OUT_DIR}/atm_scaler.pkl')

In [ ]:
#  labels + summary
import os
import numpy as np

np.save(f'{OUT_DIR}/y_train.npy', y_tr)
np.save(f'{OUT_DIR}/y_val.npy',   y_val)
np.save(f'{OUT_DIR}/y_test.npy',  y_te)

total_size = sum(
    os.path.getsize(f'{OUT_DIR}/{f}')
    for f in os.listdir(OUT_DIR)
    if f.endswith('.npy')
) / 1e9

print(f"Saved to: {OUT_DIR}")
print(f"Total .npy size: {total_size:.2f} GB")
print(f"Train: {len(y_tr):,}  | dust rate: {y_tr.mean()*100:.1f}%")
print(f"Val:   {len(y_val):,} | dust rate: {y_val.mean()*100:.1f}%")
print(f"Test:  {len(y_te):,}  | dust rate: {y_te.mean()*100:.1f}%")
print(f"Atm features: {N_ATM_VARS} | Surf features: {N_SURF_VARS}")

# Copy to Drive and Download

In [ ]:
import shutil
drive_path = f'{OUT_DIR}/sahel_enriched_preprocessed.npz'
shutil.copy2(LOCAL_PATH, drive_path)
print(f"Copied to Drive: {drive_path}")
print(f"Size: {os.path.getsize(drive_path) / 1e9:.2f} GB")

from google.colab import files
files.download(LOCAL_PATH)